### Variable Selection: Zuur 2010 VIF Threshold

In [4]:
import geopandas as gpd
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def calculate_vif_and_correlations(
    gpkg_path, 
    variables_to_exclude, 
    vif_threshold=10, 
    high_corr_threshold=0.7, 
    plot=False,
    verbose=True,
):
    """
    Calculates Variance Inflation Factor (VIF) and identifies high correlation pairs among variables.
    
    Parameters:
    - gpkg_path (str): Path to the GeoPackage file.
    - variables_to_exclude (list): List of column names to exclude from analysis.
    - vif_threshold (float): Threshold above which variables are removed based on VIF.
    - high_corr_threshold (float): Correlation coefficient above which variables are considered highly correlated.
    - plot (bool): If True, displays a heatmap of the final correlation matrix.
    
    Returns:
    - final_vif_data (pd.DataFrame): VIF values of the remaining variables.
    - removed_variables (list): Variables removed during VIF reduction.
    - final_removed_pairs (list): Variables removed due to high correlation.
    """
    
    # Load GeoPackage
    gdf = gpd.read_file(gpkg_path)
    
    # Drop excluded columns and geometry column
    data = gdf.drop(columns=variables_to_exclude, errors='ignore')
    
    # Drop any non-numeric columns
    data = data.select_dtypes(include=['number'])
    
    # Iterative VIF calculation
    def calculate_vif_df(df):
        vif_data = pd.DataFrame()
        vif_data["Variable"] = df.columns
        vif_data["VIF"] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
        return vif_data
    
    # Initial VIF calculation and reduction
    vif_data = calculate_vif_df(data)
    removed_variables = []
    while vif_data["VIF"].max() > vif_threshold:
        max_vif_var = vif_data.loc[vif_data["VIF"].idxmax(), "Variable"]
        removed_variables.append((max_vif_var, vif_data["VIF"].max()))
        data = data.drop(columns=[max_vif_var])
        vif_data = calculate_vif_df(data)
    
    if verbose:
        print("\nSequential VIF Reduction Summary:")
        for var, vif in removed_variables:
            print(f"Removed {var} with VIF = {vif:.2f}")
    
    # Iterative High Correlation Removal
    final_removed_pairs = []
    while True:
        final_correlation_matrix = data.corr().abs()
        # Select upper triangle of correlation matrix
        upper = final_correlation_matrix.where(np.triu(np.ones(final_correlation_matrix.shape), k=1).astype(bool))
        # Find index of feature columns with correlation greater than high_corr_threshold
        high_cor_pairs = [(row, col, upper.loc[row, col]) 
                          for row in upper.index 
                          for col in upper.columns 
                          if upper.loc[row, col] > high_corr_threshold]
        
        if not high_cor_pairs:
            break  # No more high correlations
        
        # Sort pairs by correlation descending
        high_cor_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
        
        # Take the pair with the highest correlation
        var1, var2, corr = high_cor_pairs[0]
        
        # Get current VIFs
        try:
            vif1 = variance_inflation_factor(data.values, data.columns.get_loc(var1))
        except:
            vif1 = np.inf  # Assign infinity if variable is missing
        try:
            vif2 = variance_inflation_factor(data.values, data.columns.get_loc(var2))
        except:
            vif2 = np.inf  # Assign infinity if variable is missing
        
        # Decide which variable to remove
        if vif1 > vif2:
            removed_var = var1
            kept_var = var2
            removed_vif = vif1
            kept_vif = vif2
        else:
            removed_var = var2
            kept_var = var1
            removed_vif = vif2
            kept_vif = vif1
        
        # Remove the variable with higher VIF
        if removed_var in data.columns:
            data = data.drop(columns=[removed_var])
            final_removed_pairs.append((removed_var, kept_var, corr, removed_vif, kept_vif))
            if verbose:
                print(f"Removed {removed_var} due to high correlation with {kept_var} "
                    f"(Correlation: {corr:.2f}, VIF: {removed_vif:.2f} vs {kept_vif:.2f})")
        else:
            if verbose:
                print(f"Variable {removed_var} already removed. Skipping.")
    
    # Plot final correlation matrix
    if plot:
        plt.figure(figsize=(10, 8))
        sns.heatmap(data.corr(), annot=True, cmap='coolwarm', center=0, fmt=".2f", linewidths=0.5)
        plt.title("Correlation Matrix Heatmap (After VIF and High Correlation Reduction)")
        plt.show()
    
    # Final VIF calculation after correlation removal
    final_vif_data = calculate_vif_df(data)
    
    # Return final VIF results, removed variables, and removed pairs
    return final_vif_data, removed_variables, final_removed_pairs


In [56]:
import geopandas as gpd

watersheds = [
            "ET sfm",
            #"LM2 sfm", "LPM sfm", "MM sfm",
            # "ET lidar", 
            #"LM2 lidar", "LPM lidar", "MM lidar"
              ]
region = "ETF"
# watersheds = [
#             #"Bennett sfm", 
#             #   "ME sfm", 
#             #   "MM sfm",
#             #   "MW sfm", 
#             #   "UE sfm", "UM sfm", "UW sfm",
#             #  "Bennett lidar",
#             #   "MM lidar","UE lidar", "UM lidar", "UW lidar"
#               ]
# region = "Bennett"
outcome_vars = [
     "ch_sfm net change mean",
    # "deposition",
    # "net"
         ]
segments = [
    # 5, 
    # 10, 
    20
    ]

VIF_threshold = 2
corr_threshold = 0.4 # Set correlation threshold for variable selection
verbose = True

import os
for segment in segments:
    for outcome_var in outcome_vars:
        for watershed in watersheds:
            if 'deposition' in outcome_var:
                type = 'deposition'
            elif 'erosion' in outcome_var:
                type = 'erosion'
            elif 'net' in outcome_var:
                type = 'net'
            else:
                raise ValueError("Outcome variable must contain 'deposition', 'erosion', or 'net'.")
            print(f"Processing {watershed} {type} data for {segment}m segments...")
            
            watershed_type = 'Combined Watersheds' if 'ET' in watershed or 'Bennett' in watershed else 'Individual Watersheds'
            gpkg_path = os.path.join(r"Y:\ATD\GIS", 
                                     region, 
                                     r"Watershed Stats\SSN2\Inputs", 
                                     f"Segmented {segment}m", 
                                     watershed_type, 
                                     f"{watershed} {type} ssn points.gpkg")
            output_formula = os.path.join(r"Y:\ATD\GIS", 
                                          region, 
                                          r"Watershed Stats\SSN2\Outputs", 
                                          f"Segmented {segment}m", 
                                          f"{watershed}_{type}_logtrans", 
                                          f"ssn_formula.txt")
            github_output_formula = os.path.join(r"C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp",
                                                 region,
                                                 "Outputs",
                                                    f"Segmented {segment}m",
                                                    f"{watershed}_{type}_logtrans",
                                                    "ssn_formula.txt")
                                                 
            os.makedirs(os.path.dirname(output_formula), exist_ok=True)
            
            #print fields in the geopackage
            print(f"Fields in {gpkg_path}:")
            gdf = gpd.read_file(gpkg_path)
            print(gdf.columns)
            gdf['ch_sfm net change mean'] = gdf['ch_sfm net change'] / gdf['ch_sfm net change count']
            # Append any other variables to exclude here
            excluded_columns = [
                'WS_ID', 'ch_distance downstream', 'ch_distance upstream', 
                'ch_ch_area', 'ch_area', 'hs_elevation range', 'ch_channel_width',
                'ch_slope upstream', 'ch_slope downstream', 'ch_watershed', 
                # 'hs_drainage_density',
                'hs_area', 'hs_aspect northness median', 'hs_aspect eastness median'
                # 'hs_accum precip mean', 'hs_mi60 mean', 'hs_storm accum 60 min mean', 'hs_storm accum 30 min mean',
                # 'hs_storm accum 10 min mean', 'ws_mi60_mean', 'ch_elevation min', 'ch_elevation mean'
            ]  

            if verbose:
                print(f"\nProcessing {watershed} {type} data...")
                print(f"Geopackage Path: {gpkg_path}")

            gdf = gpd.read_file(gpkg_path)

            variables_to_exclude = [col for col in gdf.columns if 'erosion' in col.lower() or 
                                        'deposition' in col.lower() or 'sfm' in col.lower() or 
                                        col == gdf.geometry.name]

            variables_to_exclude += [col for col in gdf.columns if 'lidar' in col.lower()]
            
            # assert that the outcome variable is in the dataframe
            assert outcome_var in gdf.columns, f"Outcome variable '{outcome_var}' not found in the GeoDataFrame."
            
            ############## COMMENT OUT BELOW FOR Multiple WS Analysis ########################################
            if 'ET' not in watershed and 'Bennett' not in  watershed:
                prefix_to_exclude = 'ws_' 
                variables_to_exclude += [col for col in gdf.columns if col.startswith(prefix_to_exclude)]
            ################################################################################################

            # Call the updated function with correct parameters
            vif_results, removed_variables, final_removed_pairs = calculate_vif_and_correlations(
                gpkg_path, 
                variables_to_exclude + excluded_columns, 
                vif_threshold=VIF_threshold, 
                high_corr_threshold=corr_threshold, 
                plot=False,
                verbose=verbose,
            )

            # Final Summary
            if verbose:
                print("\nSummary of Variables Removed Due to VIF and High Correlation:")

                print("\nVariables Removed in Sequential VIF Reduction:")
                for var, vif in removed_variables:
                    print(f"Removed {var} with VIF = {vif:.2f}")

                print("\nVariables Removed Due to High Correlation:")
                for var1, var2, corr, vif1, vif2 in final_removed_pairs:
                    print(f"Removed {var1} (VIF: {vif1:.2f}) due to high correlation with {var2} (VIF: {vif2:.2f}), correlation = {corr:.2f}")

            print(f"\nVariables After VIF threshold < {VIF_threshold}:\n")

            for variable in vif_results["Variable"]:
                
                # check if variable is the final entry in the list
                if variable == vif_results["Variable"].iloc[-1]:
                    variable = variable.replace(' ', '.')
                    variable = variable.replace('-', '.')
                    print(f'"{variable}"')
                else:
                    variable = variable.replace(' ', '.')
                    variable = variable.replace('-', '.')
                    print(f'"{variable} +",')

            # Save the formula to a text file
            with open(output_formula, 'w') as f:
                f.write(outcome_var.replace(' ', '.') + ' ~ ' + ' + '.join(vif_results["Variable"].str.replace(' ', '.').tolist()))
                f.write('\n')
            
            #copy the formula to the github folder
            with open(github_output_formula, 'w') as f:
                f.write(outcome_var.replace(' ', '.') + ' ~ ' + ' + '.join(vif_results["Variable"].str.replace(' ', '.').tolist()))
                f.write('\n')
    
    print(f"Wrote formula to {github_output_formula}")


Processing ET sfm net data for 20m segments...
Fields in Y:\ATD\GIS\ETF\Watershed Stats\SSN2\Inputs\Segmented 20m\Combined Watersheds\ET sfm net ssn points.gpkg:
Index(['elevation min', 'WS_ID', 'ws_mi60_mean', 'ws_accum_precip_mean',
       'ws_10-min storm accum_mean', 'ws_30-min storm accum_mean',
       'ws_60-min storm accum_mean', 'ws_sbs_mean', 'ws_dnbr BAER_mean',
       'ws_dnbr extended_mean', 'ws_flow accum_max', 'ws_slope_mean',
       'ws_aspect northness_mean', 'ws_aspect eastness_mean',
       'ws_slope northness_mean', 'ws_slope eastness_mean',
       'ws_bare earth_mean', 'ws_ndvi max_mean', 'ws_ndvi mean_mean',
       'ws_ndvi min_mean', 'ws_ndvi range_mean', 'ws_RV Sand', 'ws_RV Silt',
       'ws_RV Clay', 'ws_Kw', 'ch_elevation min', 'ch_watershed',
       'ch_valley_width', 'ch_area', 'ch_channel_width',
       'ch_channel width over valley width', 'ch_ch_area', 'ch_elevation mean',
       'ch_slope median', 'ch_flow accumulation max', 'ch_curvature median',
      

AssertionError: Outcome variable 'ch_sfm net change mean' not found in the GeoDataFrame.

### Belloni 2014 with LassoCV

In [40]:
import geopandas as gpd
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import RobustScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output (optional)
warnings.filterwarnings("ignore")

def calculate_vif(df):
    """
    Calculate Variance Inflation Factor (VIF) for each variable in the dataframe.
    """
    vif_data = pd.DataFrame()
    vif_data["Variable"] = df.columns
    vif_data["VIF"] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return vif_data

def remove_control_vars_highly_correlated_with_treatment(data, control_vars, treatment_var, corr_threshold):
    """
    Remove control variables that are highly correlated with the treatment variable.
    
    Parameters:
    - data: pandas DataFrame containing the data.
    - control_vars: list of control variable names.
    - treatment_var: name of the treatment variable.
    - corr_threshold: correlation threshold to identify high correlations.
    
    Returns:
    - List of control variables after removing those highly correlated with the treatment variable.
    """
    if treatment_var not in data.columns:
        raise ValueError(f"Treatment variable '{treatment_var}' not found in the data.")
    
    # Compute absolute correlations between control_vars and treatment_var
    correlations = data[control_vars].corrwith(data[treatment_var]).abs()
    
    # Identify control_vars with correlation above threshold
    high_corr_vars = correlations[correlations >= corr_threshold].index.tolist()
    
    if high_corr_vars:
        print(f"Removing control variables highly correlated with treatment variable '{treatment_var}' (corr >= {corr_threshold}): {high_corr_vars}")
        control_vars = [var for var in control_vars if var not in high_corr_vars]
    else:
        print(f"No control variables are highly correlated with treatment variable '{treatment_var}' (corr >= {corr_threshold}).")
    
    return control_vars

def remove_highly_correlated_control_vars(data, control_vars, corr_threshold, vif_threshold):
    """
    Iteratively remove control variables that are highly correlated with each other based on VIF and correlation thresholds.
    
    Parameters:
    - data: pandas DataFrame containing the data.
    - control_vars: list of control variable names.
    - corr_threshold: correlation threshold to identify high correlations.
    - vif_threshold: VIF threshold to identify multicollinearity.
    
    Returns:
    - List of control variables after removing highly correlated ones.
    """
    selected_vars = control_vars.copy()
    while True:
        X_selected = data[selected_vars]
        vif = calculate_vif(X_selected)
        
        # Compute absolute correlation matrix
        corr_matrix = X_selected.corr().abs()
        
        # Select upper triangle of correlation matrix
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
        # Find pairs with correlation greater than or equal to threshold
        high_corr_pairs = [(row, col) for row in upper.index for col in upper.columns if upper.loc[row, col] >= corr_threshold]
        
        if not high_corr_pairs:
            break  # Exit loop if no highly correlated pairs
        
        vars_to_remove = set()
        for var1, var2 in high_corr_pairs:
            vif1 = vif[vif['Variable'] == var1]['VIF'].values[0]
            vif2 = vif[vif['Variable'] == var2]['VIF'].values[0]
            if vif1 > vif2:
                vars_to_remove.add(var1)
            else:
                vars_to_remove.add(var2)
        
        if not vars_to_remove:
            break  # Exit if no variables to remove
        
        print(f"Removing variables due to high correlation (corr >= {corr_threshold}): {list(vars_to_remove)}")
        selected_vars = [var for var in selected_vars if var not in vars_to_remove]
        
        # Additionally, check for VIF threshold
        vif = calculate_vif(data[selected_vars])
        high_vif = vif[vif["VIF"] > vif_threshold]["Variable"].tolist()
        if high_vif:
            print(f"Removing variables with high VIF (> {vif_threshold}): {high_vif}")
            selected_vars = [var for var in selected_vars if var not in high_vif]
        
        if not selected_vars:
            print("All control variables removed due to high correlations or high VIF.")
            break
    
    return selected_vars

def double_selection_lasso(
    gdf, 
    variables_to_exclude, 
    outcome_var, 
    treatment_var, 
    excluded_columns=[], 
    cv=5, 
    max_iter=10000, 
    tol=1e-4, 
    random_state=42, 
    plot_corr=False, 
    VIF_threshold=3, 
    corr_threshold=0.99
):
    """
    Implements the Double Selection method using Lasso for variable selection as per Belloni et al. (2014).
    Additionally removes control variables that are highly correlated with each other or with the treatment variable.
    
    Parameters:
    - gdf: GeoDataFrame containing the data.
    - variables_to_exclude: list of variables to exclude based on initial criteria.
    - outcome_var: name of the outcome variable.
    - treatment_var: name of the treatment variable.
    - excluded_columns: additional columns to exclude.
    - cv: number of cross-validation folds for Lasso.
    - max_iter: maximum number of iterations for Lasso.
    - tol: tolerance for optimization in Lasso.
    - random_state: random state for reproducibility.
    - plot_corr: whether to plot the correlation matrix of selected variables.
    - VIF_threshold: VIF threshold to identify multicollinearity.
    - corr_threshold: correlation threshold to identify high correlations.
    
    Returns:
    - model: fitted OLS regression model.
    - selected_vars: list of selected control variables.
    - high_corr: DataFrame of high correlations among selected variables and treatment variable.
    """
    
    # Combine variables to exclude
    exclude_vars = set(variables_to_exclude + excluded_columns)
    
    # Ensure outcome and treatment variables are not excluded
    exclude_vars.discard(outcome_var)
    exclude_vars.discard(treatment_var)
    
    # Drop excluded columns and geometry column
    data = gdf.drop(columns=exclude_vars, errors='ignore')
    
    # Drop the geometry column if present
    if 'geometry' in data.columns:
        data = data.drop(columns=['geometry'])
    
    # Drop any non-numeric columns
    data = data.select_dtypes(include=['number'])
    
    # Ensure outcome and treatment variables are in the data
    if outcome_var not in data.columns:
        raise ValueError(f"Outcome variable '{outcome_var}' not found in the data.")
    if treatment_var not in data.columns:
        raise ValueError(f"Treatment variable '{treatment_var}' not found in the data.")
    
    # Define control variables
    control_vars = [col for col in data.columns if col not in [outcome_var, treatment_var]]
    
    # Step 1: Remove control variables highly correlated with the treatment variable
    print("\nStep 1: Removing control variables highly correlated with the treatment variable...")
    control_vars = remove_control_vars_highly_correlated_with_treatment(
        data, 
        control_vars, 
        treatment_var, 
        corr_threshold=corr_threshold
    )
    
    if not control_vars:
        print("No control variables remain after removing those highly correlated with the treatment variable.")
    else:
        print(f"Control variables after Step 1: {control_vars}")
    
    # Step 2: Remove highly correlated control variables based on VIF and correlation
    print("\nStep 2: Removing highly correlated control variables based on VIF and correlation...")
    control_vars = remove_highly_correlated_control_vars(
        data, 
        control_vars, 
        corr_threshold=corr_threshold, 
        vif_threshold=VIF_threshold
    )
    
    if not control_vars:
        print("No control variables remain after Step 2.")
    else:
        print(f"Control variables after Step 2: {control_vars}")
    
    X_controls = data[control_vars]
    Y = data[outcome_var]
    D = data[treatment_var]
    
    # Calculate VIF and remove highly collinear variables
    vif = calculate_vif(X_controls)
    high_vif = vif[vif["VIF"] > VIF_threshold]["Variable"].tolist()
    if high_vif:
        print(f"\nRemoving highly collinear variables (VIF > {VIF_threshold}): {high_vif}")
        X_controls = X_controls.drop(columns=high_vif)
        control_vars = [col for col in control_vars if col not in high_vif]
    
    if not control_vars:
        print("No control variables remain after removing high VIF variables.")
    
    # Standardize variables for Lasso using RobustScaler to mitigate the effect of outliers
    scaler_X = RobustScaler()
    if not X_controls.empty:
        X_controls_scaled = scaler_X.fit_transform(X_controls)
    else:
        X_controls_scaled = np.array([]).reshape(len(data), 0)  # Empty array with correct number of rows
    
    # Standardize the treatment variable
    scaler_D = RobustScaler()
    D_scaled = scaler_D.fit_transform(D.values.reshape(-1, 1)).flatten()
    
    # Lasso for Outcome Model: Y ~ X_controls
    if not X_controls_scaled.size == 0:
        lasso_outcome = LassoCV(cv=cv, random_state=random_state, max_iter=max_iter, tol=tol).fit(X_controls_scaled, Y)
        selected_outcome = np.array(control_vars)[lasso_outcome.coef_ != 0].tolist()
        print(f"\nVariables selected for Outcome Model (Y ~ X): {selected_outcome}")
    else:
        selected_outcome = []
        print("\nNo control variables available for Outcome Model.")
    
    # Lasso for Treatment Model: D ~ X_controls
    if not X_controls_scaled.size == 0:
        lasso_treatment = LassoCV(cv=cv, random_state=random_state, max_iter=max_iter, tol=tol).fit(X_controls_scaled, D_scaled)
        selected_treatment = np.array(control_vars)[lasso_treatment.coef_ != 0].tolist()
        print(f"Variables selected for Treatment Model (D ~ X): {selected_treatment}")
    else:
        selected_treatment = []
        print("No control variables available for Treatment Model.")
    
    # Union of selected variables
    selected_vars = list(set(selected_outcome + selected_treatment))
    print(f"\nCombined Selected Control Variables: {selected_vars}")
    
    if not selected_vars:
        print("No control variables selected. Proceeding with treatment variable only.")
        X_final = pd.DataFrame({'const': 1, treatment_var: D})
    else:
        # Calculate VIF for selected variables and remove if necessary
        vif_selected = calculate_vif(data[selected_vars])
        high_vif_selected = vif_selected[vif_selected["VIF"] > VIF_threshold]["Variable"].tolist()
        if high_vif_selected:
            print(f"\nRemoving selected variables with high VIF (VIF > {VIF_threshold}): {high_vif_selected}")
            selected_vars = [var for var in selected_vars if var not in high_vif_selected]
        
        if not selected_vars:
            print("All selected control variables removed due to high VIF. Proceeding with treatment variable only.")
            X_final = pd.DataFrame({'const': 1, treatment_var: D})
        else:
            # Prepare the final regression data
            X_final = data[selected_vars]
            X_final = sm.add_constant(X_final)  # Adds intercept term
            X_final = pd.concat([X_final, D], axis=1)
    
    # Fit the final OLS regression: Y ~ D + X_selected
    model = sm.OLS(Y, X_final).fit()
    
    print("\nFinal OLS Regression Results:")
    print(model.summary())
    
    # Compute the final correlation matrix
    if selected_vars:
        corr_matrix_final = data[selected_vars + [treatment_var]].corr()
        high_corr = corr_matrix_final[(corr_matrix_final >= corr_threshold) | (corr_matrix_final <= -corr_threshold)]
        
        # Drop self-correlations (set diagonal elements to NaN to omit them)
        np.fill_diagonal(high_corr.values, np.nan)
        
        # Drop rows and columns with all NaN values (to remove non-correlated variables)
        high_corr = high_corr.dropna(how='all', axis=0).dropna(how='all', axis=1)
    else:
        high_corr = None
    
    # Optional: Plot correlation matrix of the final selected variables
    if selected_vars and plot_corr:
        if high_corr is not None and not high_corr.empty:
            plt.figure(figsize=(10, 8))
            sns.heatmap(high_corr, annot=True, cmap='coolwarm', center=0, fmt=".2f", linewidths=0.5)
            plt.title(f"Correlation Matrix for Selected Variables (Treatment Variable: '{treatment_var}')")
            plt.show()
        else:
            print(f"No high correlations (|corr| >= {corr_threshold}) found among selected variables.")
    
    return model, selected_vars, high_corr

def run_double_selection(outcome_var, treatment_vars, excluded_columns, plot_corr=False, VIF_threshold=3, corr_threshold=0.7):
        # Exclude all erosion and deposition fields except for the outcome variable
    outcome_variables_to_exclude = [col for col in gdf.columns 
                                    if ('erosion' in col.lower() or 
                                        'deposition' in col.lower() or 
                                        'sfm' in col.lower()) and 
                                        col.lower() != outcome_var.lower()]

    variables_to_exclude = outcome_variables_to_exclude + excluded_columns
    
    # Initialize lists to store results
    regression_results = {}
    selected_variables_all = {}
    high_corr_all = {}

    # Loop through each treatment variable
    for treatment_var in treatment_vars:
        print(f"\n{'='*80}\nProcessing Treatment Variable: '{treatment_var}'\n{'='*80}")
        try:
            model, selected_vars, high_corr = double_selection_lasso(
                gdf=gdf,
                variables_to_exclude=variables_to_exclude,
                outcome_var=outcome_var,
                treatment_var=treatment_var,
                excluded_columns=excluded_columns,
                cv=5,
                max_iter=10000,  # Increased max_iter
                random_state=42,
                plot_corr=plot_corr,
                VIF_threshold=VIF_threshold,
                corr_threshold=corr_threshold
            )
            regression_results[treatment_var] = model
            selected_variables_all[treatment_var] = selected_vars
            high_corr_all[treatment_var] = high_corr
        except ValueError as e:
            print(f"Error: {e}")

    # Final Summary
    print(f"\n{'='*80}\nSummary of Results for All Treatment Variables\n{'='*80}")
    sorted_treatments = sorted(
            [(treatment_var, abs(model.params[treatment_var])) for treatment_var, model in regression_results.items()],
            key=lambda x: x[1], reverse=True
            )

    for treatment_var, coef_abs in sorted_treatments:
        if treatment_var in regression_results:
            model = regression_results[treatment_var]
            selected_vars = selected_variables_all[treatment_var]
            print(f"\n--- Treatment Variable: {treatment_var} ---")
            print(f"Selected Control Variables ({len(selected_vars)}): {selected_vars}")
            print(f"Treatment Effect Estimate:")
            print(f"Coefficient for '{treatment_var}': {model.params[treatment_var]:.4f}")
            print(f"P-value: {model.pvalues[treatment_var]:.4f}")
            if high_corr_all[treatment_var] is not None:
                if not high_corr_all[treatment_var].empty:
                    print(f"High Correlation Pairs for Treatment Variable '{treatment_var}':")
                    # Iterate through the upper triangle of the high correlation matrix to avoid duplicate pairs
                    for i, row in high_corr_all[treatment_var].iterrows():
                        for j, value in row.items():
                            if pd.notna(value) and high_corr_all[treatment_var].columns.get_loc(j) > high_corr_all[treatment_var].index.get_loc(i):
                                # Print only if j is after i in the index order
                                print(f"Correlation between '{i}' and '{j}': {value:.2f}")
            else:
                print(f"No high correlations among selected control variables (> {corr_threshold}) found for treatment variable '{treatment_var}'.")
    
            treatment_var = treatment_var.replace(" ", ".")
            print(f"\n{treatment_var} +")
            for var in selected_vars:
                var = var.replace(" ", ".")
                print(f"{var} +")
        else:
            print(f"\n--- Treatment Variable: {treatment_var} ---")
            print("No results available due to previous errors.")


### Currently using Belloni 2014 with correlation filtering (< 0.7)  

In [ ]:

# Load GeoPackage
gdf = gpd.read_file(gpkg_path)

plot_corr = False # Set to True if you want to plot correlation matrices
VIF_threshold = np.inf # Set VIF threshold for variable selection


excluded_columns = [
        'geometry', 'ch_distance upstream', 'ch_distance downstream', 'WS_ID', 'ch_watershed',
        'ch_slope downstream', 'ch_slope upstream', 'hs_elevation range', 'ch_area'
]  

# Define treatment variables
treatment_vars =[
    # 'ws_mi60_mean',
    # 'hs_dnbr median',
    # 'ch_stream power',
    # 'ch_channel.width.over.valley.width',
    # 'ch_stream.power.central.diff',
    # 'ws_bare_earth_mean',
]

treatment_vars = [col for col in gdf.columns 
                                if ('erosion' not in col.lower() and 
                                    'deposition' not in col.lower() and 
                                    'sfm' not in col.lower()) and
                                    col.lower() != outcome_var.lower() and
                                    col not in excluded_columns]
print(f"Treatment Variables: {treatment_vars}")

run_double_selection(outcome_var, treatment_vars, excluded_columns, plot_corr=plot_corr, 
                     VIF_threshold=VIF_threshold, corr_threshold=corr_threshold)

        

In [35]:
import pandas as pd
import re
from io import StringIO
import os
# Define the input and output file paths
input_file = r"Y:\ATD\GIS\Bennett\Watershed Stats\Aggregate Stats\SSN2\Outputs\ME_logtrans\ME_ch_sfm.erosion.norm VIF-2 corr0,6.txt"  # Replace with your actual input file path
output_excel = r"Y:\ATD\GIS\Bennett\Watershed Stats\Aggregate Stats\SSN2\Outputs\ME_logtrans\ME_output.xlsx"      # Replace with your desired output Excel file path

# Define the fixed file path to be added to each row
file_path = input_file
watershed = os.path.basename(input_file).split('_')[0]

# Read all lines from the input text file
with open(input_file, 'r') as f:
    lines = f.readlines()

# Step 1: Extract the Model Formula to get the Independent Variable
model_formula_marker = '> print(model_formula)'
independent_variable = 'Unknown'  # Default value in case extraction fails

for i, line in enumerate(lines):
    if model_formula_marker in line:
        # The model formula may span multiple lines ending with '+'
        formula_lines = []
        j = i + 1
        while j < len(lines):
            current_line = lines[j].strip()
            formula_lines.append(current_line)
            if not current_line.endswith('+') and not current_line.endswith('\\'):
                break
            j += 1
        # Combine the lines and extract the dependent variable (left side of '~')
        model_formula_full = ' '.join(formula_lines)
        dependent_var = model_formula_full.split('~')[0].strip()
        # Replace dots with spaces for readability
        independent_variable = dependent_var.replace('.', ' ')
        break

print(f"Independent Variable: {independent_variable}")

# Step 2: Locate and Extract the 'tidy' Table
tidy_marker = '> print(tidy(ssn_mod, conf.int = TRUE))'
tidy_start_index = None

for i, line in enumerate(lines):
    if tidy_marker in line:
        tidy_start_index = i
        break

if tidy_start_index is None:
    raise ValueError("The 'tidy' section was not found in the input file.")

# Find the start of the table
table_start = None
for i in range(tidy_start_index, len(lines)):
    if lines[i].startswith('# A tibble'):
        table_start = i
        break

if table_start is None:
    raise ValueError("The start of the tibble table was not found.")

# The header is typically two lines after '# A tibble' line
header_line = lines[table_start + 1].strip()
separator_line = lines[table_start + 2].strip()

print(f"Header: {header_line}")
# Extract data rows until the next prompt '>'
data_rows = []
for line in lines[table_start + 3:]:
    if line.startswith('>') or line.strip() == '':
        break
    data_rows.append(line.strip())

# Combine data rows into a single string for pandas to read
data_str = '\n'.join(data_rows)

#add header_line to the beginning of the data_str
data_str = header_line + '\n' + data_str
# Step 3: Parse the Table into a DataFrame
# Use StringIO to simulate a file-like object for pandas
df = pd.read_csv(StringIO(data_str), delim_whitespace=True)

# Verify that all expected columns are present
expected_columns = ['term', 'estimate', 'std.error', 'statistic', 'p.value', 'conf.low', 'conf.high']
if not all(col in df.columns for col in expected_columns):
    print(f"Columns present: {df}")
    raise ValueError("The table format is unexpected. Please check the input file.")

# Step 4: Reformat the DataFrame to Match Desired Output
df_formatted = pd.DataFrame({
    'Watershed': watershed,
    'Independent Variable': independent_variable,
    'Dependent Variable': df['term'],
    'estimate': df['estimate'],
    'conf.low': df['conf.low'],
    'conf.high': df['conf.high'],
    'p.value': df['p.value'],
    'File path': file_path
})

#remove rows with a p value less than 0.10
df_formatted = df_formatted[df_formatted['p.value'] < 0.10]

# Optional: Rearrange the columns in a specific order
df_formatted = df_formatted[[
    'Watershed',
    'Independent Variable',
    'Dependent Variable',
    'estimate',
    'conf.low',
    'conf.high',
    'p.value',
    'File path'
]]

# Step 5: Export the DataFrame to an Excel File
df_formatted.to_excel(output_excel, index=False)

print(f"Data has been successfully extracted and saved to '{output_excel}'.")


Independent Variable: ch_sfm erosion norm
Header: term                               estimate std.error statistic p.value conf.low conf.high
Data has been successfully extracted and saved to 'Y:\ATD\GIS\Bennett\Watershed Stats\Aggregate Stats\SSN2\Outputs\ME_logtrans\ME_output.xlsx'.
